In [0]:
from pyspark.sql import functions as F

# 1. مسار الملف بعد رفعه في Volume على Databricks
# (تأكدي من نسخ المسار الفعلي من Catalog بعد الرفع)
file_path = "/Volumes/workspace/default/raw_data/openfda_ndc_20261102T155044Z.json"

# 2. قراءة ملف الـ JSON الخام مع تفعيل multiLine=true
raw_df = spark.read.option("multiLine", "true").json(file_path)

# 3. استخراج الـ 100 سجل دواء من جوه data.results مع الاحتفاظ ببيانات المصدر (Provenance)
bronze_ndc_df = (
    raw_df.select(
        F.col("metadata.source_id").alias("source_id"),
        F.col("metadata.source_name").alias("source_name"),
        F.col("metadata.source_endpoint").alias("source_endpoint"),
        F.col("metadata.ingestion_timestamp_utc").alias("ingestion_timestamp_utc"),
        F.col("metadata.run_id").alias("run_id"),
        F.explode("data.results").alias("drug")
    )
    .select(
        "source_id",
        "source_name",
        "source_endpoint",
        "ingestion_timestamp_utc",
        "run_id",
        "drug.*"
    )
    .withColumn("bronze_loaded_at_utc", F.current_timestamp())
)

# 4. التأكد إن العدد 100 سجل وعرض البيانات
print(f"Total Bronze NDC Records: {bronze_ndc_df.count()}")
display(bronze_ndc_df.limit(10))

# 5. حفظ البيانات كـ Delta Table رسمي في طبقة البرونز
bronze_ndc_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_openfda_ndc")

print("✅ Saved Delta Table: workspace.default.bronze_openfda_ndc")